## (Trained using Kaggle Notebook)

The format might be differ, but results and workflows are expected and configured beforehand

### LSTM Conclusion

The baseline LSTM was trained across three chronological development folds, followed by a 16-trial Optuna hyperparameter search. Although Trial 14 achieved the lowest mean validation pinball loss of 0.344689 during tuning, retraining with its selected configuration produced a mean loss of 0.350222, slightly worse than the baseline's 0.349828. Therefore, the baseline configuration was retained for subsequent internal-test evaluation, as the tuned configuration did not demonstrate a consistent improvement.

In [1]:

import os
import gc
import json
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    DEVICE = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
else:
    DEVICE = torch.device("cpu")

torch.set_num_threads(min(4, os.cpu_count() or 4))

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

KAGGLE_INPUT_DIR = Path("/kaggle/input")

if not KAGGLE_INPUT_DIR.exists():
    raise FileNotFoundError("Kaggle input directory not found.")

all_npz_files = list(KAGGLE_INPUT_DIR.rglob("*.npz"))

print("\nNPZ files discovered:", len(all_npz_files))

def find_lstm_file(fold_name, file_suffix):
    matches = [
        path for path in all_npz_files
        if fold_name.lower() in path.name.lower()
        and path.stem.lower().endswith(file_suffix.lower())
    ]

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one file for {fold_name} "
            f"with suffix '{file_suffix}', found {len(matches)}:\n"
            + "\n".join(str(path) for path in matches)
        )

    return matches[0]


FOLD_NAMES = [
    "development_fold_1",
    "development_fold_2",
    "development_fold_3",
    "internal_test",
    "official_evaluation"
]

LSTM_TRAIN_FILES = {}
LSTM_EVALUATION_FILES = {}

for fold_name in FOLD_NAMES:
    LSTM_TRAIN_FILES[fold_name] = find_lstm_file(
        fold_name,
        "_train"
    )

    LSTM_EVALUATION_FILES[fold_name] = find_lstm_file(
        fold_name,
        "_evaluation"
    )

print("\nLocated LSTM datasets:")

for fold_name in FOLD_NAMES:
    print(f"\n{fold_name}")
    print("Training:", LSTM_TRAIN_FILES[fold_name])
    print("Evaluation:", LSTM_EVALUATION_FILES[fold_name])


EXPECTED_KEYS = {
    "historical_numeric",
    "historical_categorical",
    "future_numeric",
    "future_categorical",
    "static_categorical",
    "target_sales",
    "series_ids",
    "origins"
}

EVALUATION_KEYS = {
    "historical_numeric",
    "historical_categorical",
    "future_numeric",
    "future_categorical",
    "static_categorical",
    "target_sales",
    "series_ids",
    "evaluation_origin"
}

def inspect_npz(path, expected_keys):
    with np.load(path, allow_pickle=False) as data:
        actual_keys = set(data.files)

        missing = expected_keys - actual_keys

        if missing:
            raise ValueError(
                f"Missing keys in {path.name}: {sorted(missing)}"
            )

        shapes = {
            key: data[key].shape
            for key in sorted(expected_keys)
        }

    return shapes


for fold_name in FOLD_NAMES:
    train_shapes = inspect_npz(
        LSTM_TRAIN_FILES[fold_name],
        EXPECTED_KEYS
    )

    evaluation_shapes = inspect_npz(
        LSTM_EVALUATION_FILES[fold_name],
        EVALUATION_KEYS
    )

    print(f"\n{'=' * 55}")
    print(f"{fold_name}")
    print("Training shapes:")

    for key, shape in train_shapes.items():
        print(f"  {key}: {shape}")

    print("Evaluation shapes:")

    for key, shape in evaluation_shapes.items():
        print(f"  {key}: {shape}")

print("\nDataset discovery and structural verification completed.")


PyTorch: 2.11.0+cu130
Device: cuda
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB

NPZ files discovered: 10

Located LSTM datasets:

development_fold_1
Training: /kaggle/input/datasets/ryancjc/lstm-salesf/lstm/processed_datasets/development_fold_1_train.npz
Evaluation: /kaggle/input/datasets/ryancjc/lstm-salesf/lstm/processed_datasets/development_fold_1_evaluation.npz

development_fold_2
Training: /kaggle/input/datasets/ryancjc/lstm-salesf/lstm/processed_datasets/development_fold_2_train.npz
Evaluation: /kaggle/input/datasets/ryancjc/lstm-salesf/lstm/processed_datasets/development_fold_2_evaluation.npz

development_fold_3
Training: /kaggle/input/datasets/ryancjc/lstm-salesf/lstm/processed_datasets/development_fold_3_train.npz
Evaluation: /kaggle/input/datasets/ryancjc/lstm-salesf/lstm/processed_datasets/development_fold_3_evaluation.npz

internal_test
Training: /kaggle/input/datasets/ryancjc/lstm-salesf/lstm/processed_datasets/internal_test_train.npz
Evaluation: /kaggle/input/

In [2]:

import joblib

LSTM_CONFIG_FILES = list(KAGGLE_INPUT_DIR.rglob("lstm_dataset_config.json"))
LSTM_PREPROCESSOR_FILES = list(KAGGLE_INPUT_DIR.rglob("lstm_preprocessors.joblib"))

if len(LSTM_CONFIG_FILES) != 1:
    raise FileNotFoundError(
        f"Expected one lstm_dataset_config.json, found {len(LSTM_CONFIG_FILES)}"
    )

if len(LSTM_PREPROCESSOR_FILES) != 1:
    raise FileNotFoundError(
        f"Expected one lstm_preprocessors.joblib, found {len(LSTM_PREPROCESSOR_FILES)}"
    )

LSTM_CONFIG_PATH = LSTM_CONFIG_FILES[0]
LSTM_PREPROCESSOR_PATH = LSTM_PREPROCESSOR_FILES[0]

with open(LSTM_CONFIG_PATH, "r", encoding="utf-8") as f:
    LSTM_CONFIG = json.load(f)

LSTM_PREPROCESSORS = joblib.load(LSTM_PREPROCESSOR_PATH)

print("Configuration path:", LSTM_CONFIG_PATH)
print("Preprocessor path:", LSTM_PREPROCESSOR_PATH)

print("\nConfiguration keys:")
print(list(LSTM_CONFIG.keys()))

print("\nPreprocessor type:", type(LSTM_PREPROCESSORS))
if isinstance(LSTM_PREPROCESSORS, dict):
    print("Preprocessor folds:", list(LSTM_PREPROCESSORS.keys()))


CATEGORICAL_INPUT_KEYS = [
    "historical_categorical",
    "future_categorical",
    "static_categorical"
]

NUMERICAL_INPUT_KEYS = [
    "historical_numeric",
    "future_numeric"
]

LSTM_FOLD_METADATA = {}

for fold_name in FOLD_NAMES:
    train_path = LSTM_TRAIN_FILES[fold_name]
    evaluation_path = LSTM_EVALUATION_FILES[fold_name]

    with np.load(train_path, allow_pickle=False) as train_data:
        train_arrays = {
            key: train_data[key]
            for key in (
                CATEGORICAL_INPUT_KEYS
                + NUMERICAL_INPUT_KEYS
                + ["target_sales", "series_ids", "origins"]
            )
        }

    with np.load(evaluation_path, allow_pickle=False) as eval_data:
        eval_arrays = {
            key: eval_data[key]
            for key in (
                CATEGORICAL_INPUT_KEYS
                + NUMERICAL_INPUT_KEYS
                + ["target_sales", "series_ids", "evaluation_origin"]
            )
        }

    metadata = {
        "train_samples": len(train_arrays["target_sales"]),
        "historical_numeric_dim": train_arrays["historical_numeric"].shape[-1],
        "historical_categorical_dim": train_arrays["historical_categorical"].shape[-1],
        "future_numeric_dim": train_arrays["future_numeric"].shape[-1],
        "future_categorical_dim": train_arrays["future_categorical"].shape[-1],
        "static_categorical_dim": train_arrays["static_categorical"].shape[-1],
        "encoder_length": train_arrays["historical_numeric"].shape[1],
        "prediction_length": train_arrays["future_numeric"].shape[1],
        "categorical_cardinalities": {},
        "categorical_max_ids": {},
        "evaluation_max_ids": {}
    }

    for key in CATEGORICAL_INPUT_KEYS:
        train_values = train_arrays[key]
        eval_values = eval_arrays[key]

        if not np.issubdtype(train_values.dtype, np.integer):
            raise TypeError(f"{fold_name}: {key} must contain integer IDs.")

        if train_values.min() < 0 or eval_values.min() < 0:
            raise ValueError(f"{fold_name}: negative category ID found in {key}.")

        train_max = int(train_values.max())
        eval_max = int(eval_values.max())

        metadata["categorical_max_ids"][key] = train_max
        metadata["evaluation_max_ids"][key] = eval_max

        if eval_max > train_max:
            raise ValueError(
                f"{fold_name}: evaluation category ID exceeds training "
                f"range for {key}: {eval_max} > {train_max}"
            )

        if key == "historical_categorical":
            axis = (0, 1)
        elif key == "future_categorical":
            axis = (0, 1)
        else:
            axis = 0

        cardinalities = []

        for feature_index in range(train_values.shape[-1]):
            feature_values = train_values[..., feature_index]
            cardinality = int(feature_values.max()) + 1
            cardinalities.append(cardinality)

        metadata["categorical_cardinalities"][key] = cardinalities

    LSTM_FOLD_METADATA[fold_name] = metadata

    print(f"\n{fold_name}")
    print("Training samples:", metadata["train_samples"])
    print(
        "Input dimensions:",
        {
            key: metadata[key]
            for key in [
                "historical_numeric_dim",
                "historical_categorical_dim",
                "future_numeric_dim",
                "future_categorical_dim",
                "static_categorical_dim",
                "encoder_length",
                "prediction_length"
            ]
        }
    )

    print("Categorical cardinalities:")
    for key, cardinalities in metadata["categorical_cardinalities"].items():
        print(f"  {key}: {cardinalities}")

    print("Evaluation maximum category IDs:")
    print(metadata["evaluation_max_ids"])

    del train_arrays, eval_arrays
    gc.collect()

print("\nAll fold metadata inspected successfully.")


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.1 when using version 1.6.1. This might lead to breaking code or inv

Configuration path: /kaggle/input/datasets/ryancjc/lstm-salesf/lstm/lstm_dataset_config.json
Preprocessor path: /kaggle/input/datasets/ryancjc/lstm-salesf/lstm/preprocessors/lstm_preprocessors.joblib

Configuration keys:
['project', 'model', 'encoder_length', 'prediction_length', 'quantiles', 'fold_cutoffs', 'evaluation_origins', 'historical_numeric_features', 'historical_categorical_features', 'future_numeric_features', 'future_categorical_features', 'static_features', 'preprocessing', 'dataset']

Preprocessor type: <class 'dict'>
Preprocessor folds: ['numerical_preprocessors', 'calendar_scalers', 'category_vocabularies']

development_fold_1
Training samples: 49200
Input dimensions: {'historical_numeric_dim': 14, 'historical_categorical_dim': 3, 'future_numeric_dim': 9, 'future_categorical_dim': 3, 'static_categorical_dim': 5, 'encoder_length': 56, 'prediction_length': 28}
Categorical cardinalities:
  historical_categorical: [8, 32, 6]
  future_categorical: [8, 32, 6]
  static_categor

In [3]:

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader


LSTM_HIDDEN_SIZE = 128
LSTM_NUM_LAYERS = 2
LSTM_DROPOUT = 0.15

LSTM_QUANTILES = [0.1, 0.5, 0.9]
LSTM_BATCH_SIZE = 256
LSTM_LEARNING_RATE = 1e-3
LSTM_WEIGHT_DECAY = 1e-5

LSTM_EMBEDDING_DIMS = {
    fold_name: {
        key: [
            min(32, max(4, int(round(cardinality ** 0.5 * 2))))
            for cardinality in cardinalities
        ]
        for key, cardinalities
        in metadata["categorical_cardinalities"].items()
    }
    for fold_name, metadata in LSTM_FOLD_METADATA.items()
}


class RetailDemandLSTM(nn.Module):

    def __init__(
        self,
        historical_numeric_dim,
        historical_categorical_cardinalities,
        future_numeric_dim,
        future_categorical_cardinalities,
        static_categorical_cardinalities,
        hidden_size=128,
        num_layers=2,
        dropout=0.15,
        quantiles=(0.1, 0.5, 0.9)
    ):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.quantiles = list(quantiles)

        def create_embeddings(cardinalities):
            embeddings = nn.ModuleList()

            for cardinality in cardinalities:
                embedding_dim = min(
                    32,
                    max(4, int(round(cardinality ** 0.5 * 2)))
                )

                embeddings.append(
                    nn.Embedding(
                        num_embeddings=cardinality,
                        embedding_dim=embedding_dim,
                        padding_idx=0
                    )
                )

            return embeddings

        self.historical_embeddings = create_embeddings(
            historical_categorical_cardinalities
        )

        self.future_embeddings = create_embeddings(
            future_categorical_cardinalities
        )

        self.static_embeddings = create_embeddings(
            static_categorical_cardinalities
        )

        historical_embedding_dim = sum(
            embedding.embedding_dim
            for embedding in self.historical_embeddings
        )

        future_embedding_dim = sum(
            embedding.embedding_dim
            for embedding in self.future_embeddings
        )

        static_embedding_dim = sum(
            embedding.embedding_dim
            for embedding in self.static_embeddings
        )

        encoder_input_dim = (
            historical_numeric_dim + historical_embedding_dim
        )

        decoder_input_dim = (
            future_numeric_dim
            + future_embedding_dim
            + static_embedding_dim
        )

        self.encoder = nn.LSTM(
            input_size=encoder_input_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.decoder = nn.LSTM(
            input_size=decoder_input_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.output_dropout = nn.Dropout(dropout)

        self.quantile_head = nn.Linear(
            hidden_size,
            len(self.quantiles)
        )

    @staticmethod
    def embed_categorical(inputs, embedding_layers):
        embedded = []

        for feature_index, embedding_layer in enumerate(
            embedding_layers
        ):
            embedded.append(
                embedding_layer(
                    inputs[..., feature_index].long()
                )
            )

        return torch.cat(embedded, dim=-1)

    def forward(
        self,
        historical_numeric,
        historical_categorical,
        future_numeric,
        future_categorical,
        static_categorical
    ):

        historical_embedded = self.embed_categorical(
            historical_categorical,
            self.historical_embeddings
        )

        future_embedded = self.embed_categorical(
            future_categorical,
            self.future_embeddings
        )

        static_embedded = self.embed_categorical(
            static_categorical,
            self.static_embeddings
        )

        encoder_input = torch.cat(
            [
                historical_numeric.float(),
                historical_embedded
            ],
            dim=-1
        )

        _, (hidden_state, cell_state) = self.encoder(
            encoder_input
        )

        static_context = static_embedded.unsqueeze(1).expand(
            -1,
            future_numeric.size(1),
            -1
        )

        decoder_input = torch.cat(
            [
                future_numeric.float(),
                future_embedded,
                static_context
            ],
            dim=-1
        )

        decoder_output, _ = self.decoder(
            decoder_input,
            (hidden_state, cell_state)
        )

        decoder_output = self.output_dropout(decoder_output)

        predictions = self.quantile_head(decoder_output)

        return predictions


def create_lstm_model(fold_name):
    metadata = LSTM_FOLD_METADATA[fold_name]

    return RetailDemandLSTM(
        historical_numeric_dim=metadata["historical_numeric_dim"],
        historical_categorical_cardinalities=(
            metadata["categorical_cardinalities"][
                "historical_categorical"
            ]
        ),
        future_numeric_dim=metadata["future_numeric_dim"],
        future_categorical_cardinalities=(
            metadata["categorical_cardinalities"][
                "future_categorical"
            ]
        ),
        static_categorical_cardinalities=(
            metadata["categorical_cardinalities"][
                "static_categorical"
            ]
        ),
        hidden_size=LSTM_HIDDEN_SIZE,
        num_layers=LSTM_NUM_LAYERS,
        dropout=LSTM_DROPOUT,
        quantiles=LSTM_QUANTILES
    )


print("LSTM architecture defined.")
print("Hidden size:", LSTM_HIDDEN_SIZE)
print("Number of layers:", LSTM_NUM_LAYERS)
print("Dropout:", LSTM_DROPOUT)
print("Quantiles:", LSTM_QUANTILES)


LSTM architecture defined.
Hidden size: 128
Number of layers: 2
Dropout: 0.15
Quantiles: [0.1, 0.5, 0.9]


In [4]:

for fold_name in [
    "development_fold_1",
    "development_fold_2",
    "development_fold_3"
]:
    metadata = LSTM_FOLD_METADATA[fold_name]

    model = create_lstm_model(fold_name).to(DEVICE)
    model.eval()

    batch_size = 4

    dummy_inputs = {
        "historical_numeric": torch.zeros(
            batch_size, 56, 14, device=DEVICE
        ),
        "historical_categorical": torch.zeros(
            batch_size, 56, 3, dtype=torch.long, device=DEVICE
        ),
        "future_numeric": torch.zeros(
            batch_size, 28, 9, device=DEVICE
        ),
        "future_categorical": torch.zeros(
            batch_size, 28, 3, dtype=torch.long, device=DEVICE
        ),
        "static_categorical": torch.zeros(
            batch_size, 5, dtype=torch.long, device=DEVICE
        )
    }

    with torch.no_grad():
        predictions = model(**dummy_inputs)

    expected_shape = (batch_size, 28, 3)

    assert predictions.shape == expected_shape, (
        f"{fold_name}: expected {expected_shape}, "
        f"got {tuple(predictions.shape)}"
    )

    parameter_count = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    print(
        f"{fold_name}: PASSED | "
        f"Output: {tuple(predictions.shape)} | "
        f"Parameters: {parameter_count:,}"
    )

    del model, dummy_inputs, predictions
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nAll development-fold forward-pass tests passed.")


development_fold_1: PASSED | Output: (4, 28, 3) | Parameters: 463,512
development_fold_2: PASSED | Output: (4, 28, 3) | Parameters: 463,512
development_fold_3: PASSED | Output: (4, 28, 3) | Parameters: 463,512

All development-fold forward-pass tests passed.


#### Configuration

In [5]:

import time
import copy

from torch.utils.data import Dataset, DataLoader


LSTM_OUTPUT_DIR = Path("/kaggle/working/lstm")
LSTM_MODEL_DIR = LSTM_OUTPUT_DIR / "models"
LSTM_RESULT_DIR = LSTM_OUTPUT_DIR / "results"
LSTM_CHECKPOINT_DIR = LSTM_OUTPUT_DIR / "checkpoints"

for directory in [
    LSTM_MODEL_DIR,
    LSTM_RESULT_DIR,
    LSTM_CHECKPOINT_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)


LSTM_MAX_EPOCHS = 30
LSTM_EARLY_STOPPING_PATIENCE = 5
LSTM_MIN_DELTA = 1e-5

LSTM_GRADIENT_CLIP = 1.0
LSTM_NUM_WORKERS = 0

LSTM_WEIGHT_DECAY = 1e-5
LSTM_LEARNING_RATE = 1e-3

LSTM_SCHEDULER_PATIENCE = 2
LSTM_SCHEDULER_FACTOR = 0.5
LSTM_MIN_LEARNING_RATE = 1e-6


class RetailDemandDataset(Dataset):

    def __init__(self, arrays):

        self.historical_numeric = torch.from_numpy(
            np.asarray(
                arrays["historical_numeric"],
                dtype=np.float32
            )
        )

        self.historical_categorical = torch.from_numpy(
            np.asarray(
                arrays["historical_categorical"],
                dtype=np.int64
            )
        )

        self.future_numeric = torch.from_numpy(
            np.asarray(
                arrays["future_numeric"],
                dtype=np.float32
            )
        )

        self.future_categorical = torch.from_numpy(
            np.asarray(
                arrays["future_categorical"],
                dtype=np.int64
            )
        )

        self.static_categorical = torch.from_numpy(
            np.asarray(
                arrays["static_categorical"],
                dtype=np.int64
            )
        )

        self.target_sales = torch.from_numpy(
            np.asarray(
                arrays["target_sales"],
                dtype=np.float32
            )
        )

        if not torch.isfinite(self.historical_numeric).all():
            raise ValueError("Non-finite historical numerical inputs.")

        if not torch.isfinite(self.future_numeric).all():
            raise ValueError("Non-finite future numerical inputs.")

        if not torch.isfinite(self.target_sales).all():
            raise ValueError("Non-finite target values.")

        if (self.target_sales < 0).any():
            raise ValueError("Negative target sales detected.")

    def __len__(self):
        return len(self.target_sales)

    def __getitem__(self, index):

        return (
            self.historical_numeric[index],
            self.historical_categorical[index],
            self.future_numeric[index],
            self.future_categorical[index],
            self.static_categorical[index],
            self.target_sales[index]
        )


class QuantileLoss(nn.Module):

    def __init__(self, quantiles):
        super().__init__()

        self.register_buffer(
            "quantiles",
            torch.tensor(quantiles, dtype=torch.float32)
        )

    def forward(self, predictions, targets):

        targets = targets.unsqueeze(-1)

        errors = targets - predictions

        quantiles = self.quantiles.view(1, 1, -1)

        losses = torch.maximum(
            quantiles * errors,
            (quantiles - 1.0) * errors
        )

        return losses.mean()


def create_lstm_dataloader(dataset, training):

    return DataLoader(
        dataset,
        batch_size=LSTM_BATCH_SIZE,
        shuffle=training,
        num_workers=LSTM_NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=False
    )


print("Training pipeline defined.")
print("Maximum epochs:", LSTM_MAX_EPOCHS)
print("Early stopping patience:", LSTM_EARLY_STOPPING_PATIENCE)
print("Batch size:", LSTM_BATCH_SIZE)
print("Learning rate:", LSTM_LEARNING_RATE)
print("Output directory:", LSTM_OUTPUT_DIR)


Training pipeline defined.
Maximum epochs: 30
Early stopping patience: 5
Batch size: 256
Learning rate: 0.001
Output directory: /kaggle/working/lstm


#### Training and validation functions

In [6]:

def load_lstm_arrays(path, evaluation=False):

    with np.load(path, allow_pickle=False) as data:

        keys = [
            "historical_numeric",
            "historical_categorical",
            "future_numeric",
            "future_categorical",
            "static_categorical",
            "target_sales"
        ]

        arrays = {
            key: data[key].copy()
            for key in keys
        }

    return arrays


def train_one_epoch(model, dataloader, optimizer, loss_function):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for batch in dataloader:

        (
            historical_numeric,
            historical_categorical,
            future_numeric,
            future_categorical,
            static_categorical,
            targets
        ) = batch

        historical_numeric = historical_numeric.to(
            DEVICE, non_blocking=True
        )
        historical_categorical = historical_categorical.to(
            DEVICE, non_blocking=True
        )
        future_numeric = future_numeric.to(
            DEVICE, non_blocking=True
        )
        future_categorical = future_categorical.to(
            DEVICE, non_blocking=True
        )
        static_categorical = static_categorical.to(
            DEVICE, non_blocking=True
        )
        targets = targets.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        predictions = model(
            historical_numeric,
            historical_categorical,
            future_numeric,
            future_categorical,
            static_categorical
        )

        loss = loss_function(predictions, targets)

        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite training loss detected.")

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=LSTM_GRADIENT_CLIP
        )

        optimizer.step()

        batch_samples = targets.size(0)

        total_loss += loss.item() * batch_samples
        total_samples += batch_samples

    return total_loss / total_samples


@torch.no_grad()
def validate_lstm(model, dataloader, loss_function):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    for batch in dataloader:

        (
            historical_numeric,
            historical_categorical,
            future_numeric,
            future_categorical,
            static_categorical,
            targets
        ) = batch

        historical_numeric = historical_numeric.to(
            DEVICE, non_blocking=True
        )
        historical_categorical = historical_categorical.to(
            DEVICE, non_blocking=True
        )
        future_numeric = future_numeric.to(
            DEVICE, non_blocking=True
        )
        future_categorical = future_categorical.to(
            DEVICE, non_blocking=True
        )
        static_categorical = static_categorical.to(
            DEVICE, non_blocking=True
        )
        targets = targets.to(DEVICE, non_blocking=True)

        predictions = model(
            historical_numeric,
            historical_categorical,
            future_numeric,
            future_categorical,
            static_categorical
        )

        loss = loss_function(predictions, targets)

        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite validation loss detected.")

        batch_samples = targets.size(0)

        total_loss += loss.item() * batch_samples
        total_samples += batch_samples

    return total_loss / total_samples


def train_lstm_fold(fold_name):

    print("\n" + "=" * 65)
    print("Training:", fold_name)
    print("=" * 65)

    start_time = time.time()

    train_arrays = load_lstm_arrays(
        LSTM_TRAIN_FILES[fold_name]
    )

    validation_arrays = load_lstm_arrays(
        LSTM_EVALUATION_FILES[fold_name],
        evaluation=True
    )

    train_dataset = RetailDemandDataset(train_arrays)
    validation_dataset = RetailDemandDataset(validation_arrays)

    del train_arrays, validation_arrays
    gc.collect()

    train_loader = create_lstm_dataloader(
        train_dataset,
        training=True
    )

    validation_loader = create_lstm_dataloader(
        validation_dataset,
        training=False
    )

    model = create_lstm_model(fold_name).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LSTM_LEARNING_RATE,
        weight_decay=LSTM_WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=LSTM_SCHEDULER_FACTOR,
        patience=LSTM_SCHEDULER_PATIENCE,
        min_lr=LSTM_MIN_LEARNING_RATE
    )

    loss_function = QuantileLoss(
        LSTM_QUANTILES
    ).to(DEVICE)

    best_validation_loss = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0

    history = []

    checkpoint_path = (
        LSTM_CHECKPOINT_DIR / f"{fold_name}_best.pt"
    )

    for epoch in range(1, LSTM_MAX_EPOCHS + 1):

        epoch_start = time.time()

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            loss_function
        )

        validation_loss = validate_lstm(
            model,
            validation_loader,
            loss_function
        )

        scheduler.step(validation_loss)

        current_lr = optimizer.param_groups[0]["lr"]

        epoch_time = time.time() - epoch_start

        improved = (
            validation_loss
            < best_validation_loss - LSTM_MIN_DELTA
        )

        if improved:

            best_validation_loss = validation_loss
            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save(
                {
                    "fold_name": fold_name,
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_validation_loss": best_validation_loss,
                    "model_config": {
                        "hidden_size": LSTM_HIDDEN_SIZE,
                        "num_layers": LSTM_NUM_LAYERS,
                        "dropout": LSTM_DROPOUT,
                        "quantiles": LSTM_QUANTILES
                    },
                    "fold_metadata": LSTM_FOLD_METADATA[fold_name]
                },
                checkpoint_path
            )

        else:
            epochs_without_improvement += 1

        history.append({
            "fold": fold_name,
            "epoch": epoch,
            "training_loss": train_loss,
            "validation_loss": validation_loss,
            "learning_rate": current_lr,
            "epoch_time_seconds": epoch_time
        })

        print(
            f"Epoch {epoch:02d}/{LSTM_MAX_EPOCHS} | "
            f"Train: {train_loss:.6f} | "
            f"Validation: {validation_loss:.6f} | "
            f"LR: {current_lr:.7f} | "
            f"Time: {epoch_time:.1f}s"
            + (" | BEST" if improved else "")
        )

        if epochs_without_improvement >= LSTM_EARLY_STOPPING_PATIENCE:

            print(
                f"\nEarly stopping at epoch {epoch}. "
                f"Best epoch: {best_epoch}"
            )
            break

    history_df = pd.DataFrame(history)

    history_path = (
        LSTM_RESULT_DIR / f"{fold_name}_training_history.csv"
    )

    history_df.to_csv(history_path, index=False)

    elapsed_time = time.time() - start_time

    result = {
        "fold": fold_name,
        "best_epoch": best_epoch,
        "best_validation_pinball_loss": best_validation_loss,
        "epochs_trained": len(history),
        "training_time_minutes": elapsed_time / 60,
        "checkpoint": str(checkpoint_path)
    }

    print("\nFold completed:")
    print(json.dumps(result, indent=2))

    del model, optimizer, scheduler
    del train_dataset, validation_dataset
    del train_loader, validation_loader
    del loss_function, history_df

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


#### Train the three development folds

In [7]:

def load_lstm_arrays(path, evaluation=False):

    with np.load(path, allow_pickle=False) as data:

        keys = [
            "historical_numeric",
            "historical_categorical",
            "future_numeric",
            "future_categorical",
            "static_categorical",
            "target_sales"
        ]

        arrays = {
            key: data[key].copy()
            for key in keys
        }

    return arrays


def train_one_epoch(model, dataloader, optimizer, loss_function):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for batch in dataloader:

        (
            historical_numeric,
            historical_categorical,
            future_numeric,
            future_categorical,
            static_categorical,
            targets
        ) = batch

        historical_numeric = historical_numeric.to(
            DEVICE, non_blocking=True
        )
        historical_categorical = historical_categorical.to(
            DEVICE, non_blocking=True
        )
        future_numeric = future_numeric.to(
            DEVICE, non_blocking=True
        )
        future_categorical = future_categorical.to(
            DEVICE, non_blocking=True
        )
        static_categorical = static_categorical.to(
            DEVICE, non_blocking=True
        )
        targets = targets.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        predictions = model(
            historical_numeric,
            historical_categorical,
            future_numeric,
            future_categorical,
            static_categorical
        )

        loss = loss_function(predictions, targets)

        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite training loss detected.")

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=LSTM_GRADIENT_CLIP
        )

        optimizer.step()

        batch_samples = targets.size(0)

        total_loss += loss.item() * batch_samples
        total_samples += batch_samples

    return total_loss / total_samples


@torch.no_grad()
def validate_lstm(model, dataloader, loss_function):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    for batch in dataloader:

        (
            historical_numeric,
            historical_categorical,
            future_numeric,
            future_categorical,
            static_categorical,
            targets
        ) = batch

        historical_numeric = historical_numeric.to(
            DEVICE, non_blocking=True
        )
        historical_categorical = historical_categorical.to(
            DEVICE, non_blocking=True
        )
        future_numeric = future_numeric.to(
            DEVICE, non_blocking=True
        )
        future_categorical = future_categorical.to(
            DEVICE, non_blocking=True
        )
        static_categorical = static_categorical.to(
            DEVICE, non_blocking=True
        )
        targets = targets.to(DEVICE, non_blocking=True)

        predictions = model(
            historical_numeric,
            historical_categorical,
            future_numeric,
            future_categorical,
            static_categorical
        )

        loss = loss_function(predictions, targets)

        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite validation loss detected.")

        batch_samples = targets.size(0)

        total_loss += loss.item() * batch_samples
        total_samples += batch_samples

    return total_loss / total_samples


def train_lstm_fold(fold_name):

    print("\n" + "=" * 65)
    print("Training:", fold_name)
    print("=" * 65)

    start_time = time.time()

    train_arrays = load_lstm_arrays(
        LSTM_TRAIN_FILES[fold_name]
    )

    validation_arrays = load_lstm_arrays(
        LSTM_EVALUATION_FILES[fold_name],
        evaluation=True
    )

    train_dataset = RetailDemandDataset(train_arrays)
    validation_dataset = RetailDemandDataset(validation_arrays)

    del train_arrays, validation_arrays
    gc.collect()

    train_loader = create_lstm_dataloader(
        train_dataset,
        training=True
    )

    validation_loader = create_lstm_dataloader(
        validation_dataset,
        training=False
    )

    model = create_lstm_model(fold_name).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LSTM_LEARNING_RATE,
        weight_decay=LSTM_WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=LSTM_SCHEDULER_FACTOR,
        patience=LSTM_SCHEDULER_PATIENCE,
        min_lr=LSTM_MIN_LEARNING_RATE
    )

    loss_function = QuantileLoss(
        LSTM_QUANTILES
    ).to(DEVICE)

    best_validation_loss = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0

    history = []

    checkpoint_path = (
        LSTM_CHECKPOINT_DIR / f"{fold_name}_best.pt"
    )

    for epoch in range(1, LSTM_MAX_EPOCHS + 1):

        epoch_start = time.time()

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            loss_function
        )

        validation_loss = validate_lstm(
            model,
            validation_loader,
            loss_function
        )

        scheduler.step(validation_loss)

        current_lr = optimizer.param_groups[0]["lr"]

        epoch_time = time.time() - epoch_start

        improved = (
            validation_loss
            < best_validation_loss - LSTM_MIN_DELTA
        )

        if improved:

            best_validation_loss = validation_loss
            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save(
                {
                    "fold_name": fold_name,
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_validation_loss": best_validation_loss,
                    "model_config": {
                        "hidden_size": LSTM_HIDDEN_SIZE,
                        "num_layers": LSTM_NUM_LAYERS,
                        "dropout": LSTM_DROPOUT,
                        "quantiles": LSTM_QUANTILES
                    },
                    "fold_metadata": LSTM_FOLD_METADATA[fold_name]
                },
                checkpoint_path
            )

        else:
            epochs_without_improvement += 1

        history.append({
            "fold": fold_name,
            "epoch": epoch,
            "training_loss": train_loss,
            "validation_loss": validation_loss,
            "learning_rate": current_lr,
            "epoch_time_seconds": epoch_time
        })

        print(
            f"Epoch {epoch:02d}/{LSTM_MAX_EPOCHS} | "
            f"Train: {train_loss:.6f} | "
            f"Validation: {validation_loss:.6f} | "
            f"LR: {current_lr:.7f} | "
            f"Time: {epoch_time:.1f}s"
            + (" | BEST" if improved else "")
        )

        if epochs_without_improvement >= LSTM_EARLY_STOPPING_PATIENCE:

            print(
                f"\nEarly stopping at epoch {epoch}. "
                f"Best epoch: {best_epoch}"
            )
            break

    history_df = pd.DataFrame(history)

    history_path = (
        LSTM_RESULT_DIR / f"{fold_name}_training_history.csv"
    )

    history_df.to_csv(history_path, index=False)

    elapsed_time = time.time() - start_time

    result = {
        "fold": fold_name,
        "best_epoch": best_epoch,
        "best_validation_pinball_loss": best_validation_loss,
        "epochs_trained": len(history),
        "training_time_minutes": elapsed_time / 60,
        "checkpoint": str(checkpoint_path)
    }

    print("\nFold completed:")
    print(json.dumps(result, indent=2))

    del model, optimizer, scheduler
    del train_dataset, validation_dataset
    del train_loader, validation_loader
    del loss_function, history_df

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


#### Train the three development folds

In [8]:

LSTM_FOLD_RESULTS = []

for fold_name in [
    "development_fold_1",
    "development_fold_2",
    "development_fold_3"
]:

    result = train_lstm_fold(fold_name)
    LSTM_FOLD_RESULTS.append(result)

    pd.DataFrame(LSTM_FOLD_RESULTS).to_csv(
        LSTM_RESULT_DIR / "development_fold_training_summary.csv",
        index=False
    )

LSTM_TRAINING_SUMMARY = pd.DataFrame(LSTM_FOLD_RESULTS)

print("\nDevelopment training summary:")
display(LSTM_TRAINING_SUMMARY)



Training: development_fold_1
Epoch 01/30 | Train: 0.344779 | Validation: 0.415877 | LR: 0.0010000 | Time: 5.3s | BEST
Epoch 02/30 | Train: 0.274359 | Validation: 0.380174 | LR: 0.0010000 | Time: 4.5s | BEST
Epoch 03/30 | Train: 0.263204 | Validation: 0.363599 | LR: 0.0010000 | Time: 4.5s | BEST
Epoch 04/30 | Train: 0.256515 | Validation: 0.357246 | LR: 0.0010000 | Time: 4.5s | BEST
Epoch 05/30 | Train: 0.252617 | Validation: 0.355078 | LR: 0.0010000 | Time: 4.6s | BEST
Epoch 06/30 | Train: 0.249539 | Validation: 0.349220 | LR: 0.0010000 | Time: 4.6s | BEST
Epoch 07/30 | Train: 0.247240 | Validation: 0.346077 | LR: 0.0010000 | Time: 4.6s | BEST
Epoch 08/30 | Train: 0.246208 | Validation: 0.339747 | LR: 0.0010000 | Time: 4.7s | BEST
Epoch 09/30 | Train: 0.243876 | Validation: 0.345655 | LR: 0.0010000 | Time: 4.7s
Epoch 10/30 | Train: 0.241964 | Validation: 0.342471 | LR: 0.0010000 | Time: 4.8s
Epoch 11/30 | Train: 0.240296 | Validation: 0.345585 | LR: 0.0005000 | Time: 4.8s
Epoch 12/30 

,fold,best_epoch,best_validation_pinball_loss,epochs_trained,training_time_minutes,checkpoint
0,development_fold_1,8,0.339747,13,1.122763,/kaggle/working/lstm/checkpoints/development_f...
1,development_fold_2,12,0.371863,17,1.468672,/kaggle/working/lstm/checkpoints/development_f...
2,development_fold_3,13,0.340175,18,1.573304,/kaggle/working/lstm/checkpoints/development_f...


#### Optuna hyperparameter tuning

In [9]:

import optuna
import shutil
import gc
import json
import random
import time

LSTM_TUNING_DIR = LSTM_OUTPUT_DIR / "tuning"
LSTM_TUNING_DIR.mkdir(parents=True, exist_ok=True)

LSTM_BASELINE_MODEL_DIR = LSTM_MODEL_DIR / "baseline"
LSTM_TUNED_MODEL_DIR = LSTM_MODEL_DIR / "tuned"

LSTM_BASELINE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
LSTM_TUNED_MODEL_DIR.mkdir(parents=True, exist_ok=True)

LSTM_BASELINE_RESULT_DIR = LSTM_RESULT_DIR / "baseline"
LSTM_TUNED_RESULT_DIR = LSTM_RESULT_DIR / "tuned"

LSTM_BASELINE_RESULT_DIR.mkdir(parents=True, exist_ok=True)
LSTM_TUNED_RESULT_DIR.mkdir(parents=True, exist_ok=True)

for fold_name in [
    "development_fold_1",
    "development_fold_2",
    "development_fold_3"
]:
    source = LSTM_CHECKPOINT_DIR / f"{fold_name}_best.pt"

    if not source.exists():
        raise FileNotFoundError(
            f"Missing baseline checkpoint: {source}"
        )

    destination = (
        LSTM_BASELINE_MODEL_DIR / f"{fold_name}_baseline.pt"
    )

    shutil.copy2(source, destination)

baseline_summary_path = (
    LSTM_RESULT_DIR / "development_fold_training_summary.csv"
)

if baseline_summary_path.exists():
    shutil.copy2(
        baseline_summary_path,
        LSTM_BASELINE_RESULT_DIR / "baseline_training_summary.csv"
    )

OPTUNA_DATABASE_PATH = LSTM_TUNING_DIR / "lstm_optuna.db"

OPTUNA_STORAGE = (
    f"sqlite:///{OPTUNA_DATABASE_PATH.as_posix()}"
)

print("Optuna version:", optuna.__version__)
print("Baseline checkpoints preserved.")
print("Baseline results preserved.")
print("Tuning database:", OPTUNA_DATABASE_PATH)


Optuna version: 5.0.0
Baseline checkpoints preserved.
Baseline results preserved.
Tuning database: /kaggle/working/lstm/tuning/lstm_optuna.db


#### Define training configuration

In [10]:

LSTM_TUNING_TRIALS = 16
LSTM_TUNING_MAX_EPOCHS = 18
LSTM_TUNING_PATIENCE = 4

LSTM_TUNING_MIN_DELTA = 1e-5
LSTM_TUNING_GRADIENT_CLIP = 1.0

LSTM_TUNING_FOLDS = [
    "development_fold_1",
    "development_fold_2",
    "development_fold_3"
]

LSTM_TUNING_SEED = 42

print("Tuning folds:", LSTM_TUNING_FOLDS)
print("Trials:", LSTM_TUNING_TRIALS)
print("Maximum epochs per fold:", LSTM_TUNING_MAX_EPOCHS)
print("Early stopping patience:", LSTM_TUNING_PATIENCE)


Tuning folds: ['development_fold_1', 'development_fold_2', 'development_fold_3']
Trials: 16
Maximum epochs per fold: 18
Early stopping patience: 4


#### Define the tuned model factory and training function

In [11]:

def set_trial_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def create_tuned_model(fold_name, params):

    metadata = LSTM_FOLD_METADATA[fold_name]

    return RetailDemandLSTM(
        historical_numeric_dim=metadata["historical_numeric_dim"],
        historical_categorical_cardinalities=(
            metadata["categorical_cardinalities"][
                "historical_categorical"
            ]
        ),
        future_numeric_dim=metadata["future_numeric_dim"],
        future_categorical_cardinalities=(
            metadata["categorical_cardinalities"][
                "future_categorical"
            ]
        ),
        static_categorical_cardinalities=(
            metadata["categorical_cardinalities"][
                "static_categorical"
            ]
        ),
        hidden_size=params["hidden_size"],
        num_layers=params["num_layers"],
        dropout=params["dropout"],
        quantiles=LSTM_QUANTILES
    )


def run_tuning_fold(fold_name, params):

    train_arrays = load_lstm_arrays(
        LSTM_TRAIN_FILES[fold_name]
    )

    validation_arrays = load_lstm_arrays(
        LSTM_EVALUATION_FILES[fold_name],
        evaluation=True
    )

    train_dataset = RetailDemandDataset(train_arrays)
    validation_dataset = RetailDemandDataset(validation_arrays)

    del train_arrays, validation_arrays
    gc.collect()

    train_loader = DataLoader(
        train_dataset,
        batch_size=params["batch_size"],
        shuffle=True,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda")
    )

    validation_loader = DataLoader(
        validation_dataset,
        batch_size=params["batch_size"],
        shuffle=False,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda")
    )

    model = create_tuned_model(
        fold_name,
        params
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=params["learning_rate"],
        weight_decay=params["weight_decay"]
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )

    loss_function = QuantileLoss(
        LSTM_QUANTILES
    ).to(DEVICE)

    best_loss = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0

    for epoch in range(1, LSTM_TUNING_MAX_EPOCHS + 1):

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            loss_function
        )

        validation_loss = validate_lstm(
            model,
            validation_loader,
            loss_function
        )

        scheduler.step(validation_loss)

        if validation_loss < best_loss - LSTM_TUNING_MIN_DELTA:

            best_loss = validation_loss
            best_epoch = epoch
            epochs_without_improvement = 0

        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= LSTM_TUNING_PATIENCE:
            break

    result = {
        "best_validation_loss": float(best_loss),
        "best_epoch": int(best_epoch),
        "epochs_trained": int(epoch)
    }

    del model, optimizer, scheduler
    del train_dataset, validation_dataset
    del train_loader, validation_loader
    del loss_function

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


#### Define the Optuna objective

In [12]:

def lstm_optuna_objective(trial):

    params = {
        "hidden_size": trial.suggest_categorical(
            "hidden_size", [64, 96, 128, 160]
        ),
        "num_layers": trial.suggest_categorical(
            "num_layers", [1, 2]
        ),
        "dropout": trial.suggest_float(
            "dropout", 0.05, 0.30
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate", 1e-4, 3e-3, log=True
        ),
        "weight_decay": trial.suggest_float(
            "weight_decay", 1e-7, 1e-3, log=True
        ),
        "batch_size": trial.suggest_categorical(
            "batch_size", [128, 256, 512]
        )
    }

    fold_losses = {}
    fold_epochs = {}

    for fold_number, fold_name in enumerate(LSTM_TUNING_FOLDS):

        set_trial_seed(
            LSTM_TUNING_SEED + fold_number
        )

        fold_result = run_tuning_fold(
            fold_name,
            params
        )

        fold_losses[fold_name] = (
            fold_result["best_validation_loss"]
        )

        fold_epochs[fold_name] = (
            fold_result["best_epoch"]
        )

        print(
            f"Trial {trial.number} | {fold_name} | "
            f"Validation loss: "
            f"{fold_result['best_validation_loss']:.6f} | "
            f"Best epoch: {fold_result['best_epoch']}"
        )

        trial.report(
            float(np.mean(list(fold_losses.values()))),
            step=fold_number
        )

    mean_loss = float(
        np.mean(list(fold_losses.values()))
    )

    trial.set_user_attr("fold_losses", fold_losses)
    trial.set_user_attr("fold_best_epochs", fold_epochs)

    print(
        f"\nTrial {trial.number} completed | "
        f"Mean validation loss: {mean_loss:.6f}"
    )

    return mean_loss


print("Optuna objective defined successfully.")


Optuna objective defined successfully.


#### Create the Optuna study

In [13]:

LSTM_STUDY = optuna.create_study(
    study_name="retail_lstm_quantile_tuning",
    storage=OPTUNA_STORAGE,
    load_if_exists=True,
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=LSTM_TUNING_SEED,
        n_startup_trials=4
    )
)

completed_trials = sum(
    trial.state == optuna.trial.TrialState.COMPLETE
    for trial in LSTM_STUDY.trials
)

remaining_trials = max(
    0,
    LSTM_TUNING_TRIALS - completed_trials
)

print("Existing completed trials:", completed_trials)
print("Additional trials:", remaining_trials)
print("Study name:", LSTM_STUDY.study_name)

if remaining_trials > 0:

    LSTM_STUDY.optimize(
        lstm_optuna_objective,
        n_trials=remaining_trials,
        gc_after_trial=True,
        show_progress_bar=True
    )

print("\nOptuna tuning completed.")
print("Total trials:", len(LSTM_STUDY.trials))


[I 2026-11-04 03:26:26,036] A new study created in RDB with name: retail_lstm_quantile_tuning


Existing completed trials: 0
Additional trials: 16
Study name: retail_lstm_quantile_tuning


  0%|          | 0/16 [00:00<?, ?it/s]

Trial 0 | development_fold_1 | Validation loss: 0.339639 | Best epoch: 11
Trial 0 | development_fold_2 | Validation loss: 0.382923 | Best epoch: 16
Trial 0 | development_fold_3 | Validation loss: 0.340222 | Best epoch: 17

Trial 0 completed | Mean validation loss: 0.354261
[I 2026-11-04 03:28:06,068] Trial 0 finished with value: 0.3542613883813222 and parameters: {'hidden_size': 96, 'num_layers': 1, 'dropout': 0.06452090304204987, 'learning_rate': 0.00190303683817358, 'weight_decay': 2.5378155082656606e-05, 'batch_size': 512}. Best is trial 0 with value: 0.3542613883813222.
Trial 1 | development_fold_1 | Validation loss: 0.391632 | Best epoch: 18
Trial 1 | development_fold_2 | Validation loss: 0.443475 | Best epoch: 18
Trial 1 | development_fold_3 | Validation loss: 0.388674 | Best epoch: 18

Trial 1 completed | Mean validation loss: 0.407927
[I 2026-11-04 03:29:54,087] Trial 1 finished with value: 0.4079270660877228 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.157

#### Save and inspect Optuna results

In [14]:

LSTM_TRIAL_RESULTS = LSTM_STUDY.trials_dataframe(
    attrs=(
        "number",
        "value",
        "params",
        "state",
        "user_attrs"
    )
)

LSTM_TRIAL_RESULTS.to_csv(
    LSTM_TUNING_DIR / "optuna_trial_results.csv",
    index=False
)

LSTM_TRIAL_RESULTS.to_json(
    LSTM_TUNING_DIR / "optuna_trial_results.json",
    orient="records",
    indent=4
)

print("Trial results saved.")

display(
    LSTM_TRIAL_RESULTS.sort_values(
        "value",
        ascending=True
    )
)

print("\nTrial states:")
print(LSTM_TRIAL_RESULTS["state"].value_counts())


Trial results saved.


,number,value,params_batch_size,params_dropout,params_hidden_size,params_learning_rate,params_num_layers,params_weight_decay,state,user_attrs_fold_best_epochs,user_attrs_fold_losses
14,14,0.344689,128,0.130019,64,0.002888,2,1.423814e-06,COMPLETE,"{'development_fold_1': 7, 'development_fold_2'...","{'development_fold_1': 0.3329145663976669, 'de..."
12,12,0.345166,128,0.065434,96,0.002213,2,1.856513e-07,COMPLETE,"{'development_fold_1': 8, 'development_fold_2'...","{'development_fold_1': 0.3334290534257889, 'de..."
6,6,0.345965,128,0.068257,128,0.002682,2,5.279725e-04,COMPLETE,"{'development_fold_1': 7, 'development_fold_2'...","{'development_fold_1': 0.3340760803222656, 'de..."
3,3,0.346121,128,0.058597,128,0.002204,2,1.084226e-06,COMPLETE,"{'development_fold_1': 6, 'development_fold_2'...","{'development_fold_1': 0.33639845669269564, 'd..."
10,10,0.346222,128,0.251988,160,0.000999,2,2.880039e-04,COMPLETE,"{'development_fold_1': 9, 'development_fold_2'...","{'development_fold_1': 0.33330489993095397, 'd..."
7,7,0.346548,128,0.094170,128,0.000806,1,1.333208e-04,COMPLETE,"{'development_fold_1': 16, 'development_fold_2...","{'development_fold_1': 0.33061756789684293, 'd..."
4,4,0.347065,128,0.264491,128,0.000659,2,1.445009e-06,COMPLETE,"{'development_fold_1': 7, 'development_fold_2'...","{'development_fold_1': 0.3367074555158615, 'de..."
13,13,0.347400,128,0.128963,96,0.001908,2,5.814220e-04,COMPLETE,"{'development_fold_1': 6, 'development_fold_2'...","{'development_fold_1': 0.33658217191696166, 'd..."
15,15,0.348212,128,0.116104,96,0.001393,2,5.001361e-07,COMPLETE,"{'development_fold_1': 10, 'development_fold_2...","{'development_fold_1': 0.33567151963710784, 'd..."
11,11,0.350135,128,0.050856,128,0.002273,2,4.026511e-05,COMPLETE,"{'development_fold_1': 7, 'development_fold_2'...","{'development_fold_1': 0.3389326512813568, 'de..."



Trial states:
state
COMPLETE    16
Name: count, dtype: int64


#### Extract and save the best configuration

In [15]:

if not any(
    trial.state == optuna.trial.TrialState.COMPLETE
    for trial in LSTM_STUDY.trials
):
    raise RuntimeError("No completed Optuna trials available.")

BEST_LSTM_PARAMS = LSTM_STUDY.best_params

LSTM_FINAL_CONFIG = {
    "hidden_size": BEST_LSTM_PARAMS["hidden_size"],
    "num_layers": BEST_LSTM_PARAMS["num_layers"],
    "dropout": BEST_LSTM_PARAMS["dropout"],
    "batch_size": BEST_LSTM_PARAMS["batch_size"],
    "learning_rate": BEST_LSTM_PARAMS["learning_rate"],
    "weight_decay": BEST_LSTM_PARAMS["weight_decay"],
    "max_epochs": LSTM_MAX_EPOCHS,
    "early_stopping_patience": LSTM_EARLY_STOPPING_PATIENCE,
    "quantiles": LSTM_QUANTILES,
    "encoder_length": 56,
    "prediction_length": 28,
    "tuning_validation_loss": LSTM_STUDY.best_value,
    "best_trial_number": LSTM_STUDY.best_trial.number,
    "number_of_completed_trials": sum(
        trial.state == optuna.trial.TrialState.COMPLETE
        for trial in LSTM_STUDY.trials
    ),
    "tuning_folds": LSTM_TUNING_FOLDS,
    "random_seed": LSTM_TUNING_SEED
}

with open(
    LSTM_TUNING_DIR / "best_hyperparameters.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        LSTM_FINAL_CONFIG,
        f,
        indent=4
    )

print("Best Optuna parameters:")
print(json.dumps(BEST_LSTM_PARAMS, indent=4))

print("\nBest validation loss:", LSTM_STUDY.best_value)
print("\nFinal configuration saved.")


Best Optuna parameters:
{
    "hidden_size": 64,
    "num_layers": 2,
    "dropout": 0.13001901350166678,
    "learning_rate": 0.0028879323085679474,
    "weight_decay": 1.4238136211948e-06,
    "batch_size": 128
}

Best validation loss: 0.3446890882651011

Final configuration saved.


#### Retrain LSTM using the best hyperparameters

In [ ]:
import torch
import torch.nn as nn


class QuantileLoss(nn.Module):

    def __init__(self, quantiles):
        super().__init__()

        self.register_buffer(
            "quantiles",
            torch.tensor(quantiles, dtype=torch.float32)
        )

    def forward(self, predictions, targets):

        quantiles = self.quantiles.view(1, 1, -1)

        errors = targets.unsqueeze(-1) - predictions

        losses = torch.maximum(
            quantiles * errors,
            (quantiles - 1.0) * errors
        )

        return losses.mean()

In [ ]:
import json
import time
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader

# Selected Optuna Trial 14 parameters
BEST_LSTM_PARAMS = {
    "hidden_size": 64,
    "num_layers": 2,
    "dropout": 0.13001901350166678,
    "learning_rate": 0.0028879323085679474,
    "weight_decay": 1.4238136211948e-06,
    "batch_size": 128
}

# Training configuration
LSTM_FINAL_MAX_EPOCHS = 30
LSTM_FINAL_EARLY_STOPPING_PATIENCE = 5
LSTM_FINAL_MIN_DELTA = 1e-5
LSTM_FINAL_LR_PATIENCE = 2
LSTM_FINAL_MIN_LR = 1e-6

# Output directories
LSTM_OUTPUT_DIR = Path("/kaggle/working/lstm")

LSTM_TUNED_MODEL_DIR = LSTM_OUTPUT_DIR / "models" / "tuned"
LSTM_TUNED_RESULT_DIR = LSTM_OUTPUT_DIR / "results" / "tuned"
LSTM_TUNED_HISTORY_DIR = LSTM_TUNED_RESULT_DIR / "training_history"

LSTM_TUNED_MODEL_DIR.mkdir(parents=True, exist_ok=True)
LSTM_TUNED_RESULT_DIR.mkdir(parents=True, exist_ok=True)
LSTM_TUNED_HISTORY_DIR.mkdir(parents=True, exist_ok=True)

# Check that baseline checkpoints are preserved
LSTM_BASELINE_MODEL_DIR = LSTM_OUTPUT_DIR / "models" / "baseline"

print("Baseline checkpoint directory:")
print(LSTM_BASELINE_MODEL_DIR)

if LSTM_BASELINE_MODEL_DIR.exists():
    baseline_files = list(LSTM_BASELINE_MODEL_DIR.glob("*.pt"))
    print("Baseline checkpoints found:", len(baseline_files))
    for path in baseline_files:
        print(" ", path.name)
else:
    print("WARNING: Baseline model directory not found.")

# Validate required objects
required_objects = [
    "RetailDemandLSTM",
    "RetailDemandDataset",
    "QuantileLoss",
    "load_lstm_arrays",
    "train_one_epoch",
    "validate_lstm",
    "LSTM_TRAIN_FILES",
    "LSTM_EVALUATION_FILES",
    "LSTM_FOLD_METADATA",
    "LSTM_QUANTILES",
    "DEVICE",
    "RANDOM_SEED"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Missing required notebook objects: {missing_objects}"
    )

# Verify fold availability
LSTM_DEVELOPMENT_FOLDS = [
    "development_fold_1",
    "development_fold_2",
    "development_fold_3"
]

for fold_name in LSTM_DEVELOPMENT_FOLDS:
    assert fold_name in LSTM_TRAIN_FILES, f"Missing training file: {fold_name}"
    assert fold_name in LSTM_EVALUATION_FILES, f"Missing validation file: {fold_name}"
    assert fold_name in LSTM_FOLD_METADATA, f"Missing metadata: {fold_name}"

    assert Path(LSTM_TRAIN_FILES[fold_name]).exists()
    assert Path(LSTM_EVALUATION_FILES[fold_name]).exists()

print("\nConfiguration checks passed.")
print("Device:", DEVICE)
print("Quantiles:", LSTM_QUANTILES)
print("Selected parameters:", BEST_LSTM_PARAMS)

Baseline checkpoint directory:
/kaggle/working/lstm/models/baseline

Configuration checks passed.
Device: cuda
Quantiles: [0.1, 0.5, 0.9]
Selected parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.13001901350166678, 'learning_rate': 0.0028879323085679474, 'weight_decay': 1.4238136211948e-06, 'batch_size': 128}


In [ ]:
def set_lstm_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def create_tuned_lstm_model(fold_name):
    metadata = LSTM_FOLD_METADATA[fold_name]

    model = RetailDemandLSTM(
        historical_numeric_dim=metadata["historical_numeric_dim"],
        historical_categorical_cardinalities=(
            metadata["categorical_cardinalities"]["historical_categorical"]
        ),
        future_numeric_dim=metadata["future_numeric_dim"],
        future_categorical_cardinalities=(
            metadata["categorical_cardinalities"]["future_categorical"]
        ),
        static_categorical_cardinalities=(
            metadata["categorical_cardinalities"]["static_categorical"]
        ),
        hidden_size=BEST_LSTM_PARAMS["hidden_size"],
        num_layers=BEST_LSTM_PARAMS["num_layers"],
        dropout=BEST_LSTM_PARAMS["dropout"],
        quantiles=LSTM_QUANTILES
    )

    return model.to(DEVICE)


def create_tuned_lstm_loaders(fold_name):
    train_arrays = load_lstm_arrays(
        LSTM_TRAIN_FILES[fold_name]
    )

    validation_arrays = load_lstm_arrays(
        LSTM_EVALUATION_FILES[fold_name],
        evaluation=True
    )

    train_dataset = RetailDemandDataset(train_arrays)
    validation_dataset = RetailDemandDataset(validation_arrays)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BEST_LSTM_PARAMS["batch_size"],
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    validation_loader = DataLoader(
        validation_dataset,
        batch_size=BEST_LSTM_PARAMS["batch_size"],
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    return train_loader, validation_loader


def train_tuned_lstm_fold(fold_name, fold_index):

    print("\n" + "=" * 65)
    print(f"Training tuned LSTM: {fold_name}")
    print("=" * 65)

    set_lstm_seed(RANDOM_SEED + fold_index)

    train_loader, validation_loader = create_tuned_lstm_loaders(
        fold_name
    )

    model = create_tuned_lstm_model(fold_name)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=BEST_LSTM_PARAMS["learning_rate"],
        weight_decay=BEST_LSTM_PARAMS["weight_decay"]
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=LSTM_FINAL_LR_PATIENCE,
        min_lr=LSTM_FINAL_MIN_LR
    )

    loss_function = QuantileLoss(LSTM_QUANTILES).to(DEVICE)

    best_validation_loss = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0

    history = []

    checkpoint_path = (
        LSTM_TUNED_MODEL_DIR / f"{fold_name}_tuned_best.pt"
    )

    start_time = time.time()

    for epoch in range(1, LSTM_FINAL_MAX_EPOCHS + 1):

        epoch_start = time.time()

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            loss_function
        )

        validation_loss = validate_lstm(
            model,
            validation_loader,
            loss_function
        )

        scheduler.step(validation_loss)

        current_lr = optimizer.param_groups[0]["lr"]

        improved = (
            validation_loss < best_validation_loss - LSTM_FINAL_MIN_DELTA
        )

        if improved:
            best_validation_loss = validation_loss
            best_epoch = epoch
            epochs_without_improvement = 0

            checkpoint = {
                "fold_name": fold_name,
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_validation_loss": best_validation_loss,
                "model_config": {
                    **BEST_LSTM_PARAMS,
                    "quantiles": list(LSTM_QUANTILES),
                    "encoder_length": 56,
                    "prediction_length": 28
                },
                "fold_metadata": LSTM_FOLD_METADATA[fold_name],
                "random_seed": RANDOM_SEED + fold_index
            }

            torch.save(checkpoint, checkpoint_path)

        else:
            epochs_without_improvement += 1

        epoch_time = time.time() - epoch_start

        history.append({
            "fold": fold_name,
            "epoch": epoch,
            "train_loss": float(train_loss),
            "validation_loss": float(validation_loss),
            "learning_rate": float(current_lr),
            "improved": improved,
            "epoch_time_seconds": epoch_time
        })

        status = " | BEST" if improved else ""

        print(
            f"Epoch {epoch:02d}/{LSTM_FINAL_MAX_EPOCHS} | "
            f"Train: {train_loss:.6f} | "
            f"Validation: {validation_loss:.6f} | "
            f"LR: {current_lr:.7f} | "
            f"Time: {epoch_time:.1f}s"
            f"{status}"
        )

        if epochs_without_improvement >= LSTM_FINAL_EARLY_STOPPING_PATIENCE:
            print(
                f"\nEarly stopping at epoch {epoch}. "
                f"Best epoch: {best_epoch}"
            )
            break

    total_time = (time.time() - start_time) / 60

    history_df = pd.DataFrame(history)

    history_path = (
        LSTM_TUNED_HISTORY_DIR / f"{fold_name}_history.csv"
    )

    history_df.to_csv(history_path, index=False)

    result = {
        "fold": fold_name,
        "best_epoch": best_epoch,
        "best_validation_pinball_loss": best_validation_loss,
        "epochs_trained": len(history),
        "training_time_minutes": total_time,
        "checkpoint": str(checkpoint_path),
        "history": str(history_path)
    }

    print("\nFold completed:")
    print(json.dumps(result, indent=4))

    return result


# Train all three development folds
LSTM_TUNED_TRAINING_RESULTS = []

for fold_index, fold_name in enumerate(
    LSTM_DEVELOPMENT_FOLDS,
    start=1
):
    result = train_tuned_lstm_fold(
        fold_name=fold_name,
        fold_index=fold_index
    )

    LSTM_TUNED_TRAINING_RESULTS.append(result)


print("\nAll tuned LSTM development folds completed.")


Training tuned LSTM: development_fold_1
Epoch 01/30 | Train: 0.308035 | Validation: 0.380192 | LR: 0.0028879 | Time: 5.7s | BEST
Epoch 02/30 | Train: 0.261901 | Validation: 0.350939 | LR: 0.0028879 | Time: 4.8s | BEST
Epoch 03/30 | Train: 0.254355 | Validation: 0.343284 | LR: 0.0028879 | Time: 4.8s | BEST
Epoch 04/30 | Train: 0.251800 | Validation: 0.337791 | LR: 0.0028879 | Time: 4.8s | BEST
Epoch 05/30 | Train: 0.247491 | Validation: 0.340936 | LR: 0.0028879 | Time: 4.7s
Epoch 06/30 | Train: 0.245385 | Validation: 0.342472 | LR: 0.0028879 | Time: 4.7s
Epoch 07/30 | Train: 0.243590 | Validation: 0.339547 | LR: 0.0014440 | Time: 4.7s
Epoch 08/30 | Train: 0.239827 | Validation: 0.340205 | LR: 0.0014440 | Time: 4.7s
Epoch 09/30 | Train: 0.237634 | Validation: 0.338460 | LR: 0.0014440 | Time: 4.8s

Early stopping at epoch 9. Best epoch: 4

Fold completed:
{
    "fold": "development_fold_1",
    "best_epoch": 4,
    "best_validation_pinball_loss": 0.3377907943725586,
    "epochs_trained":

In [ ]:
# Save training summary
LSTM_TUNED_SUMMARY = pd.DataFrame([
    {
        key: value
        for key, value in result.items()
        if key != "history"
    }
    for result in LSTM_TUNED_TRAINING_RESULTS
])

summary_path = (
    LSTM_TUNED_RESULT_DIR / "tuned_lstm_training_summary.csv"
)

LSTM_TUNED_SUMMARY.to_csv(summary_path, index=False)

# Save selected hyperparameters and training configuration
LSTM_TUNED_CONFIG = {
    "model": "RetailDemandLSTM",
    "tuning_study": "retail_lstm_quantile_tuning",
    "best_trial": 14,
    "best_validation_pinball_loss": 0.3446890882651011,
    "hyperparameters": BEST_LSTM_PARAMS,
    "quantiles": list(LSTM_QUANTILES),
    "encoder_length": 56,
    "prediction_length": 28,
    "max_epochs": LSTM_FINAL_MAX_EPOCHS,
    "early_stopping_patience": LSTM_FINAL_EARLY_STOPPING_PATIENCE,
    "early_stopping_min_delta": LSTM_FINAL_MIN_DELTA,
    "scheduler": {
        "type": "ReduceLROnPlateau",
        "factor": 0.5,
        "patience": LSTM_FINAL_LR_PATIENCE,
        "minimum_learning_rate": LSTM_FINAL_MIN_LR
    },
    "development_folds": LSTM_TUNED_TRAINING_RESULTS
}

config_path = (
    LSTM_TUNED_RESULT_DIR / "tuned_lstm_config.json"
)

with open(config_path, "w") as file:
    json.dump(LSTM_TUNED_CONFIG, file, indent=4)

print("Tuned LSTM training summary:")
display(LSTM_TUNED_SUMMARY)

print("\nSaved files:")
print(summary_path)
print(config_path)
print(LSTM_TUNED_MODEL_DIR)
print(LSTM_TUNED_HISTORY_DIR)

Tuned LSTM training summary:


,fold,best_epoch,best_validation_pinball_loss,epochs_trained,training_time_minutes,checkpoint
0,development_fold_1,4,0.337791,9,0.725483,/kaggle/working/lstm/models/tuned/development_...
1,development_fold_2,9,0.371847,14,1.145056,/kaggle/working/lstm/models/tuned/development_...
2,development_fold_3,5,0.341026,10,0.827618,/kaggle/working/lstm/models/tuned/development_...



Saved files:
/kaggle/working/lstm/results/tuned/tuned_lstm_training_summary.csv
/kaggle/working/lstm/results/tuned/tuned_lstm_config.json
/kaggle/working/lstm/models/tuned
/kaggle/working/lstm/results/tuned/training_history


In [ ]:
print("Verifying tuned LSTM checkpoints...\n")

for fold_name in LSTM_DEVELOPMENT_FOLDS:

    checkpoint_path = (
        LSTM_TUNED_MODEL_DIR / f"{fold_name}_tuned_best.pt"
    )

    assert checkpoint_path.exists(), (
        f"Missing checkpoint: {checkpoint_path}"
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False
    )

    model = create_tuned_lstm_model(fold_name)

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    print(
        f"{fold_name}: PASSED | "
        f"Best epoch: {checkpoint['epoch']} | "
        f"Validation loss: "
        f"{checkpoint['best_validation_loss']:.6f}"
    )

    del model
    del checkpoint

print("\nAll three tuned checkpoints passed verification.")

Verifying tuned LSTM checkpoints...

development_fold_1: PASSED | Best epoch: 4 | Validation loss: 0.337791
development_fold_2: PASSED | Best epoch: 9 | Validation loss: 0.371847
development_fold_3: PASSED | Best epoch: 5 | Validation loss: 0.341026

All three tuned checkpoints passed verification.
